In [ ]:
# IMPORTS
import pandas as pd
from emag_reports.logger import logger
from emag_reports.models.prediction import ClassificationScoring
from emag_reports.models.prediction import PredictionSettings
from emag_reports.prediction.classification import ClassificationManager
from emag_reports.prediction.common import print_tree
from emag_reports.prediction.summary import ClassificationSummarizer
from emag_reports.preprocess import balance_decision_classes
from emag_reports.preprocess import remove_low_correlation_attributes
from emag_reports.preprocess import encode_dataset
from emag_reports.preprocess import decode_dataset
from emag_reports.preprocess import impute_missing_values
from emag_reports.preprocess import remove_low_quality_attributes
from IPython.display import display
from IPython.display import HTML
from IPython.display import Markdown as md
from itables import show
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler

In [ ]:
## DEFAULT PARAMETERS
# DATASET PARAMETERS
TITLE = "Predictive analysis report - classification"
NAME = "diabetes"
CLASS_ATTRIBUTE = "class"
# DATA PREPARATION
# Unimportant attributes elimination
UNIMPORTANT_ATTRIBUTE_ELIMINATION = True
USE_LOW_CORRELATION_ATTRIBUTE_REMOVAL = True
I = 0.9
S = 0.9
M = 0.9
T = 0.9
# Missing values imputation
MISSING_VALUES_IMPUTATION = True
# Strategy
STRATEGY_NUMERICAL = 'mean'
STRATEGY_NOMINAL = 'most_frequent'
# Decision class balancing
DECISION_CLASS_BALANCING = True
# ALGORITHMS PARAMETERS
ANALYSIS_MODE = PredictionSettings.AnalysisMode.TRAIN_TEST
TEST_SIZE = 0.3
# GRID SEARCH
GRID_SEARCH = True
SCORING = ClassificationScoring.BALANCED_ACCURACY
# INCLUDED MODELS
LOGISTIC_REGRESSION = True
DECISION_TREE = True
RANDOM_FOREST = True
KNN = True
NAIVE_BAYES = True
SVM = True
GRADIENT_BOOSTING = True

DATASET = None

In [ ]:
# Error flags
PREPROCESSING_ERROR = False

In [ ]:
display_analysis_mode = "train-test split" if ANALYSIS_MODE == PredictionSettings.AnalysisMode.TRAIN_TEST else "5-fold cross-validation"
html_content = f"""
<div style='text-align: center; margin-bottom: 30px; font-family: Arial, sans-serif;'>
    <h1 style='font-size: 2em; color: #2c3e50;'><strong>Predictive analysis report</strong></h1>
    <h2 style='font-size: 1.3em; color: #34495e;'>Evaluate the predictive ability of various models</h2>
</div>
<div style='margin: 0 auto; width: 90%; font-family: Arial, sans-serif; color: #2c3e50;'>
    <section style='margin-bottom: 30px;'>
        <h2 style='border-bottom: 2px solid #7f8c8d; padding-bottom: 10px;'>Report information</h2>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p><strong>Report name:</strong> <span style='font-style: italic; color: #2980b9;'>{TITLE}</span></p>
            <p><strong>Dataset name:</strong> <span style='font-style: italic; color: #2980b9;'>{NAME}</span></p>
        </div>
    </section>
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Data preprocessing parameters</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Model evaluation method: <span style='font-style: italic; color: #2980b9;'>{display_analysis_mode}</span></p>"""

if ANALYSIS_MODE == PredictionSettings.AnalysisMode.TRAIN_TEST:
    html_content += f"""
            <p>Test set size: <span style='font-style: italic; color: #2980b9;'>{TEST_SIZE}</span></p>"""

html_content += f"""
            <p>Perform a search of hyperparameter space: <span style='font-style: italic; color: #2980b9;'>{GRID_SEARCH}</span></p>
            <p>Remove attributes of low quality: <span style='font-style: italic; color: #2980b9;'>{UNIMPORTANT_ATTRIBUTE_ELIMINATION}</span></p>"""

if UNIMPORTANT_ATTRIBUTE_ELIMINATION:
    html_content += f"""
            <div style='padding-left: 2em;'>
                <p>Remove attributes weakly correlated with the decision attribute: <span style='font-style: italic; color: #2980b9;'>{USE_LOW_CORRELATION_ATTRIBUTE_REMOVAL}</span></p>
                <p>Excessive number of unique values threshold: <span style='font-style: italic; color: #2980b9;'>{I}</span></p>
                <p>Stability threshold: <span style='font-style: italic; color: #2980b9;'>{S}</span></p>
                <p>Missing values threshold: <span style='font-style: italic; color: #2980b9;'>{M}</span></p>
                <p>Text-like - the variable values resemble text threshold: <span style='font-style: italic; color: #2980b9;'>{T}</span></p>
            </div>"""

html_content += f"""
            <p>Replace missing values: <span style='font-style: italic; color: #2980b9;'>{MISSING_VALUES_IMPUTATION}</span></p>"""

if MISSING_VALUES_IMPUTATION:
    html_content += f"""
            <div style='padding-left: 2em;'>
                <p>For numeric attributes by: <span style='font-style: italic; color: #2980b9;'>{STRATEGY_NUMERICAL}</span></p>
                <p>For nominal attributes by: <span style='font-style: italic; color: #2980b9;'>{STRATEGY_NOMINAL}</span></p>
            </div>"""

html_content += f"""
            <p>Balance the sizes of decision classes: <span style='font-style: italic; color: #2980b9;'>{DECISION_CLASS_BALANCING}</span></p>
        </div>
    </section>"""

html_content += f"""
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Selected models</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Logistic regression: <span style='font-style: italic; color: #2980b9;'>{LOGISTIC_REGRESSION}</span></p>
            <p>Trees: <span style='font-style: italic; color: #2980b9;'>{DECISION_TREE}</span></p>
            <p>K nearest neighbors: <span style='font-style: italic; color: #2980b9;'>{KNN}</span></p>
            <p>Naive Bayes model: <span style='font-style: italic; color: #2980b9;'>{NAIVE_BAYES}</span></p>
            <p>Support vector machine: <span style='font-style: italic; color: #2980b9;'>{SVM}</span></p>
            <p>Random forest: <span style='font-style: italic; color: #2980b9;'>{RANDOM_FOREST}</span></p>
            <p>Gradient boosted trees: <span style='font-style: italic; color: #2980b9;'>{GRADIENT_BOOSTING}</span></p>
        </div>
    </section>
</div>
"""

display(HTML(html_content))

In [ ]:
try:
    # Load dataset
    df = pd.read_json(DATASET)
    df.index.name = "index"
    classes = df[CLASS_ATTRIBUTE].value_counts()
    samples_per_class = min(classes.min(), 5)
    sample = df.groupby(CLASS_ATTRIBUTE, group_keys=False).apply(lambda x: x.sample(samples_per_class))
    selected_indices = sample.index
    sample_will_change = DECISION_CLASS_BALANCING or not MISSING_VALUES_IMPUTATION
    display(md(f"## Dataset sample"))
    show(sample.reset_index(drop=True), show_index=False)
    
    # Preprocessing
    df.columns = [str(col) for col in df.columns]
    original_features = df.columns.drop(CLASS_ATTRIBUTE)
    display(md("## Preprocessed dataset"))
    
    if UNIMPORTANT_ATTRIBUTE_ELIMINATION:
        low_correlation_attribute = {}
        low_quality_details = {}
        if USE_LOW_CORRELATION_ATTRIBUTE_REMOVAL: 
            df, low_correlation_attributes = remove_low_correlation_attributes(df, CLASS_ATTRIBUTE)
        ISMT_THRESHOLD = {'I': I, 'S': T, 'M': M, 'T': T}
        df, low_quality_details = remove_low_quality_attributes(df, CLASS_ATTRIBUTE, ISMT_THRESHOLD)
        display(md("#### Removed attributes"))
        reasons_combined = {**low_correlation_attributes, **low_quality_details}
        for reason, details in reasons_combined.items():
            if details is not None and len(details) > 0:
                if isinstance(details, pd.Index):
                    details = details.tolist()  
                details_str = ', '.join(details)
            else:
                details_str = 'None'
            display(md(f"**{reason}:** {details_str}"))
    
    if MISSING_VALUES_IMPUTATION:
        df = impute_missing_values(df, CLASS_ATTRIBUTE, STRATEGY_NUMERICAL, STRATEGY_NOMINAL)
    else:
        df = df.dropna()
    
    if DECISION_CLASS_BALANCING: 
        encoded_df, label_encoder, onehot_encoder, categorical_cols = encode_dataset(df, CLASS_ATTRIBUTE)
        encoded_df, DECISION_CLASS_BALANCING = balance_decision_classes(encoded_df, CLASS_ATTRIBUTE)
        df = decode_dataset(encoded_df, df, onehot_encoder, CLASS_ATTRIBUTE, categorical_cols, label_encoder)

    if sample_will_change:
        sample = df.groupby(CLASS_ATTRIBUTE, group_keys=False).apply(lambda x: x.sample(samples_per_class))
        selected_indices = sample.index
        display(md("NOTE: Due to decision class balancing and/or removal of missing values, preprocessed and encoded samples will contain different examples than the initial sample."))
    
    display(md("#### Dataset after preprocessing"))
    preprocessed_sample = df.loc[selected_indices].reset_index(drop=True) 
    show(preprocessed_sample, show_index=False)
    
    encoded_df, label_encoder, onehot_encoder, categorical_cols = encode_dataset(df, CLASS_ATTRIBUTE)
    
    display(md("#### Dataset after one-hot encoding and label encoding"))
    encoded_sample = encoded_df.loc[selected_indices].reset_index(drop=True)
    show(encoded_sample, show_index=False)
    df = encoded_df
    
    # Split dataset
    X, y = df.drop(CLASS_ATTRIBUTE, axis=1), df[CLASS_ATTRIBUTE]
    features = X.columns
    class_names = label_encoder.classes_
    # SCALING
    scaler = StandardScaler()
    X = pd.DataFrame(scaler.fit_transform(X), columns=features, index=X.index)
    # TT / CV SPLIT
    if ANALYSIS_MODE == PredictionSettings.AnalysisMode.TRAIN_TEST:
        split = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=42)
    else:
        split = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # IS BINARY
    BINARY = df[CLASS_ATTRIBUTE].nunique() == 2
    # LIST OF MODELS FOR LATER COMPARISON
    models = []
    
    common_params = {
        "scoring": SCORING,
        "grid_search": GRID_SEARCH,
        "X": X,
        "y": y,
        "split": split,
        "mode": ANALYSIS_MODE,
        "class_names": class_names,
        "original_features": original_features,
        "binary": BINARY,
    }
except Exception as e:
    logger.exception(e)
    display(md("An error occurred during preprocessing. No models will be trained."))
    PREPROCESSING_ERROR = True

In [ ]:
def show_coefs(regression_model):
    if BINARY:
        columns = ["Coefficients"]
        coefs = pd.DataFrame(regression_model.coef_.T, index=features, columns=columns)
        intercept = pd.DataFrame(regression_model.intercept_.T, index=["intercept"], columns=columns)
    else:
        columns = class_names
        coefs = pd.DataFrame(regression_model.coef_.T, index=features, columns=columns)
        intercept = pd.DataFrame(regression_model.intercept_, index=class_names, columns=["intercept"]).T
    coefs = pd.concat([coefs, intercept])
    display(md(f"#### Coefficients:"))
    show(coefs)


if LOGISTIC_REGRESSION and not PREPROCESSING_ERROR:
    from sklearn.linear_model import LogisticRegression
    model_class = LogisticRegression
    model_name = "Logistic regression"
    param_grid = {
        "tol": [0.001, 0.01, 0.1, ],
        "C": [0.01, 0.1, 1, ],
    }
    fixed_grid = {"max_iter": 1000}
    
    manager = ClassificationManager(
        model_class=model_class,
        model_name=model_name,
        param_grid=param_grid,
        fixed_grid=fixed_grid,
        **common_params,
        additional_method=show_coefs,
    )
    manager.run()
    models.append(manager)

In [ ]:
def show_tree(tree_model):
    print_tree(tree_model, features, class_names)


if DECISION_TREE and not PREPROCESSING_ERROR:
    from sklearn.tree import DecisionTreeClassifier
    model_class = DecisionTreeClassifier
    model_name = "Decision trees"
    param_grid = {
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 3, 5, 10],
        "min_samples_split": [0.01, 0.02, 0.05],
    }
    fixed_grid = {}
    
    manager = ClassificationManager(
        model_class=model_class,
        model_name=model_name,
        param_grid=param_grid,
        fixed_grid=fixed_grid,
        **common_params,
        additional_method=show_tree,
    )
    manager.run()
    models.append(manager)

In [ ]:
if RANDOM_FOREST and not PREPROCESSING_ERROR:
    from sklearn.ensemble import RandomForestClassifier
    model_class = RandomForestClassifier
    model_name = "Random forests"
    param_grid = {
        "n_estimators": [100, 1000],
    }
    fixed_grid = {}
    
    manager = ClassificationManager(
        model_class=model_class,
        model_name=model_name,
        param_grid=param_grid,
        fixed_grid=fixed_grid,
        **common_params,
    )
    manager.run()
    models.append(manager)

In [ ]:
if KNN and not PREPROCESSING_ERROR:
    from sklearn.neighbors import KNeighborsClassifier
    model_class = KNeighborsClassifier
    model_name = "K-nearest neighbors"
    param_grid = {
        "n_neighbors": [1, 2, 5, 10, 20, 50],
        "algorithm": ["auto"],
    }
    fixed_grid = {}
    
    manager = ClassificationManager(
        model_class=model_class,
        model_name=model_name,
        param_grid=param_grid,
        fixed_grid=fixed_grid,
        **common_params,
    )
    manager.run()
    models.append(manager)

In [ ]:
if NAIVE_BAYES and not PREPROCESSING_ERROR:
    from sklearn.naive_bayes import GaussianNB
    model_class = GaussianNB
    model_name = "Naive Bayes classifier"
    param_grid = {}
    fixed_grid = {}
    
    manager = ClassificationManager(
        model_class=model_class,
        model_name=model_name,
        param_grid=param_grid,
        fixed_grid=fixed_grid,
        **common_params,
    )
    manager.run()
    models.append(manager)

In [ ]:
if SVM and not PREPROCESSING_ERROR:
    from sklearn.svm import SVC
    model_class = SVC
    model_name = "Radial SVM"
    param_grid = {
        "C": [0.01, 0.1],
        "gamma": [0.01, 0.1],
        "kernel": ["rbf"],
    }
    fixed_grid = {
        "probability": True,
    }
    
    manager = ClassificationManager(
        model_class=model_class,
        model_name=model_name,
        param_grid=param_grid,
        fixed_grid=fixed_grid,
        **common_params,
    )
    manager.run()
    models.append(manager)

In [ ]:
if GRADIENT_BOOSTING and not PREPROCESSING_ERROR:
    from xgboost import XGBClassifier
    model_class = XGBClassifier
    model_name = "XGBoost"
    param_grid = {
        "booster": ["gbtree"],
        "n_estimators": [100, 200, 500],
        "learning_rate": [0.3, 0.4],
    }
    fixed_grid = {}
    
    manager = ClassificationManager(
        model_class=model_class,
        model_name=model_name,
        param_grid=param_grid,
        fixed_grid=fixed_grid,
        **common_params,
    )
    manager.run()
    models.append(manager)

In [ ]:
if not PREPROCESSING_ERROR and len(models):
    summary = ClassificationSummarizer(models, mode=ANALYSIS_MODE, binary=BINARY)
    summary.compare()